# Kilosort Spike Raster Plot - Example Usage

This notebook demonstrates how to load kilosort spike data and create raster plots aligned to the Open Ephys recording synchronization frame.

## Overview

Kilosort outputs spike times in samples (at 20kHz typically). This code:
1. Loads spike times and cluster assignments from kilosort output
2. Filters for 'good' units (from cluster_group.tsv)
3. Aligns spike times to OE recording start (ms from start, starting at 0)
4. Creates raster plots using Bokeh (consistent with existing codebase)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add project to path if needed
project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing.kilosort_loader import (
    load_and_align_kilosort_spikes,
    load_kilosort_params,
    get_good_cluster_ids
)
from eye_tracking_system_tools.preprocessing.kilosort_visualization import (
    plot_spike_raster,
    plot_spike_raster_by_cluster
)

## 1. Initialize BlockSync Object

Load the block you want to analyze. For PV_208 block 19:

In [ ]:
# Configuration
experiment_path = Path(r"D:\sample_data_for_eye_repo")
animal = "PV_208"
experiment_date = "2025_12_14"
block_num = "019"

# Channel mapping (from batch_block_synchronization.ipynb)
channeldict = {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"}

# Create BlockSync object
block = BlockSync(
    animal_call=animal,
    experiment_date=experiment_date,
    block_num=block_num,
    path_to_animal_folder=str(experiment_path),
    channeldict=channeldict
)

print(f"Block path: {block.block_path}")
print(f"OE path: {block.oe_path}")
print(f"Sample rate: {block.sample_rate} Hz")
print(f"OE recording start time: {block.oe_rec.globalStartTime_ms:.2f} ms")

## 2. Load and Align Kilosort Spike Data

The `load_and_align_kilosort_spikes` function:
- Automatically finds the kilosort folder (spikeSorting/kilosort within oe_path)
- Loads spike times and cluster assignments
- Filters for 'good' units (optional)
- Aligns spike times to OE recording start (ms from start, starting at 0)

In [ ]:
# Load and align spike data
spike_times_ms, spike_clusters, metadata = load_and_align_kilosort_spikes(
    block=block,
    kilosort_path=None,  # Auto-detect from block.oe_path
    filter_good_only=True  # Only return spikes from 'good' clusters
)

print(f"\nLoaded spike data:")
print(f"  Total spikes (before filtering): {metadata['n_spikes_total']:,}")
print(f"  Spikes returned (good units): {metadata['n_spikes_returned']:,}")
print(f"  Number of good clusters: {metadata['n_good_clusters']}")
print(f"  Good cluster IDs: {metadata['good_cluster_ids']}")
print(f"\nSpike time range:")
print(f"  Min: {np.min(spike_times_ms):.2f} ms")
print(f"  Max: {np.max(spike_times_ms):.2f} ms")
print(f"  Duration: {np.max(spike_times_ms) - np.min(spike_times_ms):.2f} ms")
print(f"\nMetadata:")
print(f"  Kilosort sample rate: {metadata['kilosort_sample_rate']} Hz")
print(f"  OE sample rate: {metadata['oe_sample_rate']} Hz")
print(f"  OE recording start (globalStartTime_ms): {metadata['oe_recording_start_ms']:.2f} ms")

## 3. Create Raster Plot

Create a raster plot showing all good units. Each cluster is plotted as a separate row.

In [ ]:
# Plot full raster (all spikes)
p = plot_spike_raster(
    block=block,
    filter_good_only=True,
    time_range_ms=None,  # Plot all spikes
    max_spikes_per_cluster=None,  # Plot all spikes (set to e.g., 10000 for performance)
    width=1200,
    height=600,
    to_browser=True
)

## 4. Plot Specific Time Range

Focus on a specific time window (e.g., first 10 seconds):

In [ ]:
# Plot first 10 seconds
p = plot_spike_raster(
    block=block,
    filter_good_only=True,
    time_range_ms=(0, 10000),  # First 10 seconds
    width=1200,
    height=600,
    to_browser=True
)

## 5. Plot Specific Clusters

Plot only specific cluster IDs:

In [ ]:
# Plot specific clusters (e.g., first 5 good clusters)
good_cluster_ids = get_good_cluster_ids(block.oe_path / 'spikeSorting' / 'kilosort')
clusters_to_plot = good_cluster_ids[:5]  # First 5 good clusters

p = plot_spike_raster(
    block=block,
    filter_good_only=False,  # We'll filter manually
    cluster_ids=clusters_to_plot,
    time_range_ms=(0, 10000),
    width=1200,
    height=400,
    to_browser=True
)

## 6. Individual Cluster Plots

Create separate plots for each cluster (useful for detailed inspection):

In [ ]:
# Plot each cluster separately
figures = plot_spike_raster_by_cluster(
    block=block,
    filter_good_only=True,
    time_range_ms=(0, 10000),  # First 10 seconds
    width=1200,
    height_per_cluster=100,
    to_browser=True
)

## 7. Verify Alignment

Check that spike times are properly aligned by comparing with OE recording duration:

In [ ]:
# Compare spike time range with OE recording duration
oe_duration_ms = block.oe_rec.recordingDuration_ms
spike_max_ms = np.max(spike_times_ms)

print(f"OE recording duration: {oe_duration_ms:.2f} ms")
print(f"Max spike time (aligned): {spike_max_ms:.2f} ms")
print(f"Difference: {abs(oe_duration_ms - spike_max_ms):.2f} ms")

if abs(oe_duration_ms - spike_max_ms) < 100:  # Within 100ms
    print("\n✓ Alignment looks good!")
else:
    print("\n⚠ Warning: Large difference detected. Check alignment.")

## 8. Export Spike Data

Save aligned spike data to CSV for use in other analyses:

In [ ]:
# Create DataFrame with aligned spike data
spike_df = pd.DataFrame({
    'spike_time_ms': spike_times_ms,
    'cluster_id': spike_clusters
})

# Sort by time
spike_df = spike_df.sort_values('spike_time_ms').reset_index(drop=True)

# Save to CSV
output_path = block.analysis_path / 'kilosort_spikes_aligned.csv'
spike_df.to_csv(output_path, index=False)
print(f"Saved aligned spike data to: {output_path}")
print(f"\nFirst few rows:")
print(spike_df.head(10))